In [1]:
import sys
!{sys.executable} -m pip install librosa tensorflow numpy pandas matplotlib seaborn scikit-learn tqdm

  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached soundfile-0.13.1-py2.py3-none-macosx_11_0_arm64.whl.metadata (16 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 MB 6.9 MB/s  0:00:32m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 8.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 5.8 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 8.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 7.4 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 9.3 MB/s  0:00:00 eta 0:00:01m
Using cached audioread-3.1.0-py3-none-any.whl (23 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 7.8 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━

In [2]:
import os
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# Deep Learning Framework
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow Version: {tf.__version__}")
print("Libraries Loaded. Ready for Phase 2 DL Implementation.")

TensorFlow Version: 2.21.0
Libraries Loaded. Ready for Phase 2 DL Implementation.


In [3]:
# Based on your image and terminal path, this is where the raw data lives.
RAW_DATA_PATH = "../data/raw/Baby Crying Sounds/"
PROCESSED_DATA_PATH = "../data/processed/"

# Ensure the processed folder exists
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Audio processing configuration (IoT friendly settings)
SAMPLE_RATE = 16000  # Downsample for faster processing/edge compatibility
DURATION = 5          # Analyze 5 seconds of audio per sample
N_MELS = 128         # Number of Mel bands (the 'height' of our image)
MAX_PAD_LEN = 157    # Calculated consistent width based on 5s @ 16kHz

def extract_spectrogram(file_path):
    """
    Loads an audio file and converts it into a Log-Mel Spectrogram.
    Standardizes the input shape for the CNN by padding/trimming.
    """
    try:
        # Load audio (mono)
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
        
        # Generate Mel Spectrogram
        spectrogram = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=N_MELS)
        
        # Convert to power (dB) - logarithmic scale is crucial for audio
        log_spec = librosa.power_to_db(spectrogram, ref=np.max)
        
        # Padding/Trimming to ensure consistent input size for the CNN (image shape)
        if log_spec.shape[1] > MAX_PAD_LEN:
            log_spec = log_spec[:, :MAX_PAD_LEN]
        else:
            pad_width = MAX_PAD_LEN - log_spec.shape[1]
            log_spec = np.pad(log_spec, pad_width=((0, 0), (0, pad_width)), mode='constant')
            
        return log_spec
    except Exception as e:
        print(f"Error encountered at file: {file_path}. Skipping.")
        return None

In [4]:
data = []
labels = []

# Loop through each subfolder (category) in your raw data directory
for folder in tqdm(os.listdir(RAW_DATA_PATH)):
    folder_path = os.path.join(RAW_DATA_PATH, folder)
    
    # Skip hidden files like .DS_Store
    if not os.path.isdir(folder_path):
        continue
        
    print(f"Processing category: {folder}")
    
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_path = os.path.join(folder_path, file)
            
            # Use the function we defined to get the Log-Mel Spectrogram
            features = extract_spectrogram(file_path)
            
            if features is not None:
                data.append(features)
                labels.append(folder)

# Convert to numpy arrays for the Deep Learning model
X = np.array(data)
y = np.array(labels)

# Add a channel dimension (needed for CNNs: height, width, channels)
X = X[..., np.newaxis]

print(f"\nFinal Data Shape: {X.shape}") # Should be (Samples, 128, 157, 1)
print(f"Unique Categories: {np.unique(y)}")

  0%|          | 0/9 [00:00<?, ?it/s]

Processing category: silence


/opt/anaconda3/envs/DevEnv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 11%|█         | 1/9 [00:20<02:43, 20.45s/it]

Processing category: discomfort


 22%|██▏       | 2/9 [00:21<01:03,  9.05s/it]

...

Processing category: belly pain


 89%|████████▉ | 8/9 [00:28<00:02,  2.02s/it]

Processing category: laugh


100%|██████████| 9/9 [00:29<00:00,  3.28s/it]


Final Data Shape: (1197, 128, 157, 1)
Unique Categories: ['belly pain' 'burping' 'cold_hot' 'discomfort' 'hungry' 'laugh' 'silence'
 'tired']


In [5]:
# Convert text labels to integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data: 80% Training, 20% Validation
# Using stratify ensures both sets have the same percentage of each cry type
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training Samples: {X_train.shape[0]}")
print(f"Validation Samples: {X_val.shape[0]}")
print(f"Classes identified: {le.classes_}")

Training Samples: 957
Validation Samples: 240
Classes identified: ['belly pain' 'burping' 'cold_hot' 'discomfort' 'hungry' 'laugh' 'silence'
 'tired']


In [6]:
num_classes = len(le.classes_)

model = models.Sequential([
    # Layer 1: Convolution + Pooling
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 157, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Layer 2: Deeper features
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Layer 3: Complex patterns
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.Dropout(0.3), # Regularization to prevent overfitting
    layers.Flatten(),
    
    # Dense Layers for classification
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax') # Softmax for multi-class
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

/opt/anaconda3/envs/DevEnv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 155, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 155, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 77, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 75, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 35, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 28, 35, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 125440)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     8,028,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,121,800 (30.98 MB)

 Trainable params: 8,121,608 (30.98 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
# Early Stopping: Stop if the model stops getting better to save time and prevent overfitting
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5,          # Wait 5 epochs for improvement before quitting
    restore_best_weights=True
)

print("Starting Training... this may take a few minutes.")

# Start the training
history = model.fit(
    X_train, y_train,
    epochs=30,           # Max epochs, though Early Stopping will likely stop it sooner
    batch_size=32,       # Standard batch size for stability
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
# Plot Training vs Validation Accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.legend()

# Plot Training vs Validation Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.legend()

plt.show()